In [3]:
import json
import os
import time
import pytz
import requests
from pathlib import Path
from datetime import datetime,timezone,timedelta
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from dotenv import load_dotenv
from bot_template import BaseBot, OrderBook, Order, OrderRequest, OrderResponse, Trade, Side, Product

load_dotenv()

EXCHANGE_URL  = "http://ec2-52-49-69-152.eu-west-1.compute.amazonaws.com/"   
USERNAME = os.getenv("EXCHANGE_USERNAME")
PASSWORD = os.getenv("EXCHANGE_PASSWORD")

LONDON_TZ = pytz.timezone('Europe/London')

In [2]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278

#伦敦15分钟天气预报, 返回一个包含以下列的DataFrame: 时间, 温度, 湿度.
def get_weather(past_steps=96, forecast_steps=96):
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "humidity": m["relative_humidity_2m"],
    })

In [4]:
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"

#获取威斯敏斯特地区泰晤士河的最新潮汐数据. 返回一个包含以下列的DataFrame: 时间, 水平
def get_thames(limit=400):
    resp = requests.get(
        f"https://environment.data.gov.uk/flood-monitoring/id/measures/{THAMES_MEASURE}/readings",
        params={"_sorted": "", "_limit": limit},
    )
    resp.raise_for_status()
    items = resp.json().get("items", [])
    df = pd.DataFrame(items)[["dateTime", "value"]].rename(columns={"dateTime": "time", "value": "level"})
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Europe/London")
    return df.sort_values("time").reset_index(drop=True)

In [ ]:
# 潮汐调和分析,包含5个主要分潮: M2(12.42h), S2(12.00h), N2(12.66h), K1(23.93h), O1(25.82h)
TIDAL_CONSTITUENTS = [
    ('M2', 2*np.pi/12.4206),
    ('S2', 2*np.pi/12.0000),
    ('N2', 2*np.pi/12.6583),
    ('K1', 2*np.pi/23.9345),
    ('O1', 2*np.pi/25.8194),
]

# 调和分析: h(t) = Z0 + sum_i [A_i*cos(w_i*t) + B_i*sin(w_i*t)]
# 用最小二乘法拟合，返回 (Z0, [(A1,B1), (A2,B2), ...]) 和残差std.
def fit_tidal_harmonic(thames_df):
    t0 = thames_df['time'].min()
    hours = (thames_df['time'] - t0).dt.total_seconds().values / 3600
    y = thames_df['level'].values
    # 构建设计矩阵: [1, cos(w1*t), sin(w1*t), cos(w2*t), sin(w2*t), ...]
    cols = [np.ones(len(hours))]
    for _, w in TIDAL_CONSTITUENTS:
        cols.append(np.cos(w * hours))
        cols.append(np.sin(w * hours))
    X = np.column_stack(cols)
    # 最小二乘法
    coeffs, residuals, _, _ = np.linalg.lstsq(X, y, rcond=None)
    y_pred = X @ coeffs
    res_std = float(np.std(y - y_pred))
    return coeffs, t0, res_std

# 用调和系数预测指定时刻的水位
def predict_tidal_level(coeffs, t0, target_time):
    h = (target_time - t0).total_seconds() / 3600
    val = coeffs[0]
    for i, (_, w) in enumerate(TIDAL_CONSTITUENTS):
        val += coeffs[1 + 2*i] * np.cos(w * h)
        val += coeffs[2 + 2*i] * np.sin(w * h)
    return float(val)

# 用调和系数批量预测一组时刻的水位
def predict_tidal_series(coeffs, t0, times):
    hours = np.array([(t - t0).total_seconds() / 3600 for t in times])
    vals = np.full(len(hours), coeffs[0])
    for i, (_, w) in enumerate(TIDAL_CONSTITUENTS):
        vals += coeffs[1 + 2*i] * np.cos(w * hours)
        vals += coeffs[2 + 2*i] * np.sin(w * hours)
    return vals


In [7]:
AERODATABOX_KEY = os.getenv("AERODATABOX_KEY")
AERODATABOX_HOST = "aerodatabox.p.rapidapi.com"
AIRPORT = "LHR"

# 按相对时间窗口(从现在开始的偏移量)获取航班信息. 该API抓取次数有限制所以说要保留历史文档
def fetch_flights(airport=AIRPORT,offset_minutes=-360,duration_minutes=720,filters:dict|None=None):
    params = f"?offsetMinutes={offset_minutes}&durationMinutes={duration_minutes}&direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

# 据明确的本地时间范围(最大跨度为12小时)获取航班信息. 该API抓取次数有限制所以说要保留历史文档
def fetch_flights_range(airport=AIRPORT,from_local="2026-02-28T12:00",to_local="2026-02-29T00:00",filters: dict|None = None):
    params = "?direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}/{from_local}/{to_local}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

In [ ]:
# 解析航班时间. 优先用实际跑道时间, 其次修订时间, 最后计划时间
def parse_flight_time(flight_record, flight_type):
    time_fields = ['runwayTime', 'revisedTime', 'scheduledTime']
    movement = flight_record['movement']
    utc_str = None
    for field in time_fields:
        if field in movement and movement[field].get('utc'):
            utc_str = movement[field]['utc']
            break
    if utc_str is None:
        raise ValueError('No valid time field found')
    dt = pytz.utc.localize(datetime.strptime(utc_str, '%Y-%m-%d %H:%MZ'))
    return dt.astimezone(LONDON_TZ)

# 按时间窗口统计到达/出发航班数
def group_flights(flights_data, start_time, end_time):
    arrivals = 0
    for arr in flights_data.get('arrivals', []):
        try:
            if start_time <= parse_flight_time(arr, 'arrival') < end_time:
                arrivals += 1
        except Exception:
            pass
    departures = 0
    for dep in flights_data.get('departures', []):
        try:
            if start_time <= parse_flight_time(dep, 'departure') < end_time:
                departures += 1
        except Exception:
            pass
    return arrivals, departures


In [ ]:
def get_lhr_count_from(d1, d2):
    return (len(d1.get('arrivals', [])) + len(d1.get('departures', [])) + len(d2.get('arrivals', [])) + len(d2.get('departures', [])))

def get_lhr_index_from(d1, d2):
    date_range = pd.date_range(
        start=LONDON_TZ.localize(datetime(2026, 2, 28, 12, 0, 0)),
        end=LONDON_TZ.localize(datetime(2026, 3, 1, 12, 0, 0)),
        freq='30min', tz='Europe/London')
    idx = 0.0
    for i in range(24):
        arr, dep = group_flights(d1, date_range[i], date_range[i+1])
        idx += ((arr - dep) / (arr + dep)) * 100 if (arr + dep) else 0
    for i in range(24):
        arr, dep = group_flights(d2, date_range[i+24], date_range[i+25])
        idx += ((arr - dep) / (arr + dep)) * 100 if (arr + dep) else 0
    return round(abs(idx))


In [4]:
class FairValueEngine:
    CACHE_TTL = 60
    SETTLE_TIME = datetime(2026, 3, 1, 12, 0, 0)
    SESSION_START = datetime(2026, 2, 28, 12, 0, 0)
    MC_SAMPLES = 20000
    TOTAL_HOURS = 24.0

    def __init__(self):
        self._weather_cache = None
        self._weather_ts = 0.0
        self._thames_cache = None
        self._thames_ts = 0.0
        self._tidal_coeffs = None
        self._flights_cache = None
        self._drift_history = {}
        self._market_weight = {}

    # 缓存, 60秒内复用同一套数据
    def _get_weather(self):
        if time.time() - self._weather_ts > self.CACHE_TTL:
            self._weather_cache = get_weather(past_steps=96, forecast_steps=96)
            self._weather_ts = time.time()
        return self._weather_cache

    # 缓存, 60秒内复用同一套数据
    def _get_thames(self):
        if time.time() - self._thames_ts > self.CACHE_TTL:
            df = get_thames(limit=672)
            self._thames_cache = df
            self._tidal_coeffs = fit_tidal_harmonic(df)
            self._thames_ts = time.time()
        return self._thames_cache, self._tidal_coeffs

    # API调用有限制, 只记录一次数据
    def init_flights(self):
        if self._flights_cache is None:
            print('Fetching flight data (2 API calls)...')
            d1 = fetch_flights_range(from_local='2026-02-28T12:00', to_local='2026-03-01T00:00')
            d2 = fetch_flights_range(from_local='2026-03-01T00:00', to_local='2026-03-01T12:00')
            self._flights_cache = (d1, d2)
            print(f'Loaded {get_lhr_count_from(d1, d2)} flights')
        return self._flights_cache

    # 这个方法的唯一目的是: 判断模型最近是否系统性地偏离市场, 并据此调整市场的信任权重
    def _update_drift(self, symbol, model_fv, market_mid):
        from collections import deque
        # 用deque(maxlen=10)滚动保留最近 10 条
        if symbol not in self._drift_history:
            self._drift_history[symbol] = deque(maxlen=10)
            self._market_weight[symbol] = 0.2
        self._drift_history[symbol].append(model_fv - market_mid)
        hist = list(self._drift_history[symbol])
        if len(hist) >= 5:
            signs = [1 if x > 0 else -1 for x in hist[-5:]]
            if abs(sum(signs)) == 5:
                self._market_weight[symbol] = min(0.7, self._market_weight[symbol] + 0.1)
            else:
                self._market_weight[symbol] = max(0.1, self._market_weight[symbol] - 0.05)

    # 这是贝叶斯推断在正态分布下的标准公式. 核心思想: 精度（方差的倒数）越高的信息源, 在融合时占的比重越大
    # 后验 FV = (精度_模型 × fv_模型 + 精度_市场 × mid_市场) / (精度_模型 + 精度_市场)
    # 市场的std被设为model_std*(2.0-w), w越大市场std越小, 市场精度越高, 后验就越向市场价靠拢
    # 这个机制让 bot 在模型持续出错时能自动修正报价，而不是死守模型。
    def bayesian_update(self, symbol, model_fv, model_std, market_mid):
        if market_mid is None:
            return model_fv, model_std
        self._update_drift(symbol, model_fv, market_mid)
        w = self._market_weight.get(symbol, 0.2)
        market_std = model_std * (2.0 - w)
        w_model = 1.0 / model_std**2
        w_market = 1.0 / market_std**2
        post_fv = (w_model * model_fv + w_market * market_mid) / (w_model + w_market)
        post_std = (1.0 / (w_model + w_market)) ** 0.5
        return post_fv, post_std

    # 工具方法, 返回距结算还有多少小时, 用于时间加权计算
    def _hours_to_settle(self):
        settle  = LONDON_TZ.localize(self.SETTLE_TIME)
        now_lon = datetime.now(timezone.utc).astimezone(pytz.timezone('Europe/London'))
        return max((settle - now_lon).total_seconds() / 3600, 0.0)

    # 计算wx_spot. 越接近结算, 天气预报越准, 不确定度按平方根收窄. 
    def wx_spot_fv(self):
        df = self._get_weather()
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        start = LONDON_TZ.localize(self.SESSION_START)
        df = df[(df['time'] >= start) & (df['time'] <= settle)].copy()
        df['temp_f'] = df['temperature'] * 9/5 + 32
        df['prod'] = df['temp_f'] * df['humidity']
        row = df[df['time'] == settle]
        if row.empty: row = df.iloc[[-1]]
        fv = float(row['prod'].values[0])
        base_std = float(df['prod'].std()) if len(df) > 1 else fv * 0.02
        h_left = self._hours_to_settle()
        time_factor = max((h_left / self.TOTAL_HOURS) ** 0.5, 0.05)
        std = base_std * time_factor
        return round(fv), max(std, 1.0)

    # 计算wx_sum. std用per_step_std×sqrt(未来步数)估算, 这是随机游走的标准误差公式, 步数越多不确定性越大.
    def wx_sum_fv(self):
        df = self._get_weather()
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        start = LONDON_TZ.localize(self.SESSION_START)
        df = df[(df['time'] >= start) & (df['time'] <= settle)].copy()
        df['temp_f'] = df['temperature'] * 9/5 + 32
        df['prod'] = df['temp_f'] * df['humidity']
        now_lon = datetime.now(timezone.utc).astimezone(pytz.timezone('Europe/London'))
        observed = float(df[df['time'] <= now_lon]['prod'].sum() / 100)
        forecast_df = df[df['time'] > now_lon]
        forecast = float(forecast_df['prod'].sum() / 100)
        fv  = observed + forecast
        per_step_std = float(df['prod'].std()) / 100 if len(df) > 1 else 5.0
        std = per_step_std * max(len(forecast_df) ** 0.5, 1)
        return round(fv), max(std, 1.0)
        
    # 计算tide_spot.
    def tide_spot_fv(self):
        df, (coeffs, t0, res_std) = self._get_thames()
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        fv = abs(predict_tidal_level(coeffs, t0, settle)) * 1000
        std = res_std * 1000
        return round(fv), max(std, 1.0)

    # 计算tide_swing.
    # std用Monte Carlo估算: 给预测水位加 1000 组随机噪声, 算出 1000 个 payoff, 取标准差. 这比解析法更准确, 因为 payoff 是非线性的（有折点）.
    def tide_swing_fv(self):
        df, (coeffs, t0, res_std) = self._get_thames()
        start = LONDON_TZ.localize(self.SESSION_START)
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        dr = pd.date_range(start=start, end=settle, freq='15min', tz='Europe/London')
        levels = predict_tidal_series(coeffs, t0, dr)
        df_sess = df[df['time'] >= start].copy()
        level_series = pd.Series(levels, index=dr)
        # 用真实观测数据覆盖已有的时间点
        for _, row in df_sess.iterrows():
            if row['time'] in level_series.index:
                level_series[row['time']] = row['level']
        diff_cm = level_series.diff().abs() * 100
        payoff = (np.maximum(0, 20 - diff_cm) + np.maximum(0, diff_cm - 25)).sum()
        fv = float(payoff)
        n_steps = len(dr)
        noise = np.random.normal(0, res_std, (1000, n_steps))
        noisy_levels = levels[np.newaxis, :] + noise
        noisy_diff = np.abs(np.diff(noisy_levels, axis=1)) * 100
        noisy_payoff = (np.maximum(0, 20 - noisy_diff) + np.maximum(0, noisy_diff - 25)).sum(axis=1)
        std = float(noisy_payoff.std())
        return round(fv), max(std, 1.0)

    # 计算lhr_count.
    def lhr_count_fv(self):
        if self._flights_cache is None: return 1400, 50.0
        d1, d2 = self._flights_cache
        return round(float(get_lhr_count_from(d1, d2))), 30.0

    # 计算lhr_index.
    def lhr_index_fv(self):
        if self._flights_cache is None: return 50, 20.0
        d1, d2 = self._flights_cache
        return round(float(get_lhr_index_from(d1, d2))), 15.0

    # 计算lon_etf. 假设独立, 方差相加.
    def lon_etf_fv(self, ts=None, wx=None, lc=None):
        if ts is None: ts = self.tide_spot_fv()
        if wx is None: wx = self.wx_spot_fv()
        if lc is None: lc = self.lhr_count_fv()
        fv  = ts[0] + wx[0] + lc[0]
        std = (ts[1]**2 + wx[1]**2 + lc[1]**2) ** 0.5
        return round(fv), max(std, 1.0)

    # 计算lon_fly. LON_FLY是一个非线性 payoff, 不能直接用解析公式, 所以用 MC:
    # TIDE_SPOT~FoldedNormal, WX_SPOT~Normal, LHR_COUNT~Poisson.
    # 生成 20000组ETF样本, 套FLY的分段线性payoff公式, 取均值和标准差.
    def lon_fly_fv(self, ts=None, wx=None, lc=None):
        if ts is None: ts = self.tide_spot_fv()
        if wx is None: wx = self.wx_spot_fv()
        if lc is None: lc = self.lhr_count_fv()
        N = self.MC_SAMPLES
        tide_samples = np.abs(np.random.normal(ts[0]/1000.0, ts[1]/1000.0, N)) * 1000
        wx_samples   = np.random.normal(wx[0], wx[1], N)
        lhr_samples  = np.random.poisson(max(lc[0], 1), N).astype(float)
        etf_samples  = np.maximum(0, tide_samples + wx_samples + lhr_samples)
        fly_samples  = (2*np.maximum(0, 6200 - etf_samples) + np.maximum(0, etf_samples - 6200) - 2*np.maximum(0, etf_samples - 6600) + 3*np.maximum(0, etf_samples - 7000))
        return round(float(fly_samples.mean())), max(float(fly_samples.std()), 1.0)

    # 用有限差分法估算delta和gamma.
    def lon_fly_greeks(self, ts=None, wx=None, lc=None):
        if ts is None: ts = self.tide_spot_fv()
        if wx is None: wx = self.wx_spot_fv()
        if lc is None: lc = self.lhr_count_fv()
        etf_fv = ts[0] + wx[0] + lc[0]
        etf_std = (ts[1]**2 + wx[1]**2 + lc[1]**2) ** 0.5
        h = max(etf_std * 0.1, 10.0)
        def fly_mc(etf_center):
            s = np.random.normal(etf_center, etf_std, self.MC_SAMPLES)
            s = np.maximum(0, s)
            return float((2*np.maximum(0,6200-s)+np.maximum(0,s-6200) -2*np.maximum(0,s-6600)+3*np.maximum(0,s-7000)).mean())
        # 固定随机种子, 保证三次 MC 用的是同一组随机数, 消除 MC 噪声对差分的干扰
        np.random.seed(42)
        f_up = fly_mc(etf_fv + h)
        np.random.seed(42)
        f_mid = fly_mc(etf_fv)
        np.random.seed(42)
        f_down = fly_mc(etf_fv - h)
        delta = (f_up - f_down) / (2 * h)
        gamma = (f_up - 2*f_mid + f_down) / (h**2)
        return delta, gamma

    # 对每个品种都跑一遍bayesian_update, 把模型FV和市场mid融合后再返回. 最终fair_values字典里存的已经是贝叶斯后验值, bot的做市和套利模块直接用这个结果
    def get_all(self, market_mids=None):
        if market_mids is None: market_mids = {}
        ts  = self.tide_spot_fv()
        tsw = self.tide_swing_fv()
        wx  = self.wx_spot_fv()
        wxs = self.wx_sum_fv()
        lc  = self.lhr_count_fv()
        li  = self.lhr_index_fv()
        etf = self.lon_etf_fv(ts=ts, wx=wx, lc=lc)
        fly = self.lon_fly_fv(ts=ts, wx=wx, lc=lc)
        raw = {'TIDE_SPOT':ts,'TIDE_SWING':tsw,'WX_SPOT':wx,'WX_SUM':wxs, 'LHR_COUNT':lc,'LHR_INDEX':li,'LON_ETF':etf,'LON_FLY':fly}
        result = {}
        for sym, (fv, std) in raw.items():
            mid = market_mids.get(sym)
            post_fv, post_std = self.bayesian_update(sym, fv, std, mid)
            result[sym] = (post_fv, post_std)
        return result

In [ ]:
from collections import deque
class ParamCalibrator:
    WARMUP_ROUNDS = 6    # 热身轮数 (约60秒), 之后开始校准
    CALIB_INTERVAL = 30   # 每30轮重新校准一次
    TARGET_TURNOVER = 0.3  # 目标: 每轮循环库存周转率 30%

    def __init__(self):
        self._round = 0
        self._fill_history = {}   # {symbol: deque of (spread_at_fill, fill_vol)}
        self._basis_buf = deque(maxlen=200)
        self._vol_buf = {}   # {symbol: deque of realized_vol}
        # 校准结果
        self.gamma = {}   # {symbol: float}
        self.k = {}   # {symbol: float}
        self.arb_thresh = 25.0
        # 默认值 (热身期使用)
        self._gamma_default = 0.1
        self._k_default = 1.5

    # 这三个方法本身不做任何计算, 只是往缓冲区塞数据, 由bot的其他模块在运行时调用
    # record_fill(symbol, spread_at_fill, fill_vol)) 每次成交时调用, 记录成交时的价差和成交量. 用于后续估算 k.
    def record_fill(self, symbol, spread_at_fill, fill_vol):
        if symbol not in self._fill_history:
            self._fill_history[symbol] = deque(maxlen=50)
        self._fill_history[symbol].append((spread_at_fill, fill_vol))

    # record_basis(basis)每次计算ETF basis(LON_ETF mid - 三腿之和)时调用, 用于校准套利阈值.
    # LON_ETF 理论上等于三条腿之和: LON_ETF = TIDE_SPOT + WX_SPOT + LHR_COUNT
    # 但市场上这四个品种独立交易, 价格会有偏差. 这个偏差就叫 basis: basis = LON_ETF市场价 - (TIDE_SPOT + WX_SPOT + LHR_COUNT)市场价
    # basis 不为零时理论上有套利机会, 但是交易有摩擦(价差、滑点), basis 本身有正常的随机波动, 所以需要一个阈值, 只有 basis 超过这个值才值得出手
    # 如果阈值设死, 可能在某些市场状态下太敏感, 频繁误触发; 在另一些状态下又太迟钝, 错过真正的机会
    # 因此_calibrate_arb_thresh 的做法是: 看历史上 basis 的绝对值, 取第 95 个百分位数作为阈值
    def record_basis(self, basis):
        self._basis_buf.append(basis)

    # record_vol(symbol, sigma)每轮主循环调用, 记录每个品种的短期波动率. 用于校准 gamma.
    def record_vol(self, symbol, sigma):
        if symbol not in self._vol_buf:
            self._vol_buf[symbol] = deque(maxlen=50)
        self._vol_buf[symbol].append(sigma)

    # 每主循环调用一次, 到达校准间隔时重新校准所有参数. 主循环每10秒调一次 tick, 所以:
    # 前60秒是热身期, 用默认参数, 让缓冲区积累足够数据
    # 第60秒时立即第一次校准, 之后每 300 秒（30轮）校准一次
    def tick(self, symbols, fair_values, mins_left):
        self._round += 1
        if self._round < self.WARMUP_ROUNDS:
            return  
        if (self._round - self.WARMUP_ROUNDS) % self.CALIB_INTERVAL != 0:
            return
        self._calibrate_gamma_k(symbols, fair_values, mins_left)
        self._calibrate_arb_thresh()

    # 校准gamma和k
    def _calibrate_gamma_k(self, symbols, fair_values, mins_left):
        # 计算剩余时间 T. T 在A-S模型里代表"还有多少时间可以平仓", T越小风险越高
        T = max(mins_left / (24 * 60), 1e-4)
        for sym in symbols:
            # 校准gamma. 优先用record_vol积累的实际波动率历史的均值. 如果还没有记录, 就用fair_values里的std作为替代.
            vols = list(self._vol_buf.get(sym, []))
            sigma = float(np.mean(vols)) if vols else fair_values.get(sym, (0, 5))[1]
            sigma = max(sigma, 1.0)
            # A-S 模型的最优半价差简化后约等于：half_spread ≈ gamma × sigma^2 × T / 2
            # 我们的目标是: 让报出来的价差等于TARGET_TURNOVER × sigma, 即0.3 × sigma
            # 从 gamma*sigma^2*T = target_spread反推gamma, gamma ∝ 1/(sigma × T)
            target_spread = self.TARGET_TURNOVER * sigma
            gamma_new = target_spread / max(sigma**2 * T, 1e-6)
            gamma_new = float(np.clip(gamma_new, 0.01, 2.0))
            # 指数移动平均平滑, 避免参数跳变
            old = self.gamma.get(sym, self._gamma_default)
            self.gamma[sym] = 0.7 * old + 0.3 * gamma_new

            # 校准k. k在A-S模型里代表市场深度, 直觉上是"报出去的单有多容易成交"
            # 这里用一个简化估算：k = 平均成交量 / (平均价差 × 2)
            # 成交量大、价差窄 → k大, 市场活跃, 可以报窄一点; 成交量小、价差宽 → k小, 市场冷清, 需要报宽一点吸引成交
            fills = list(self._fill_history.get(sym, []))
            if len(fills) >= 3:
                avg_fill_vol = float(np.mean([f[1] for f in fills]))
                avg_spread = float(np.mean([f[0] for f in fills if f[0] > 0]))
                k_new = avg_fill_vol / max(avg_spread * 2, 1.0)
                k_new = float(np.clip(k_new, 0.3, 10.0))
                old_k = self.k.get(sym, self._k_default)
                self.k[sym] = 0.7 * old_k + 0.3 * k_new
            else:
                self.k.setdefault(sym, self._k_default)

        print(f'[CALIB] gamma={{{", ".join(f"{s}:{v:.3f}" for s,v in self.gamma.items())}}}')
        print(f'[CALIB] k={{{", ".join(f"{s}:{v:.2f}" for s,v in self.k.items())}}}')

    # 套利阈值校准
    # 把历史basis的95th percentile作为套利触发阈值. 意思是"只有 basis 超过历史上 95% 的情况才触发套利", 这样正常的basis波动不会误触发
    def _calibrate_arb_thresh(self):
        if len(self._basis_buf) < 20:
            return
        arr = np.array(self._basis_buf)
        p95 = float(np.percentile(np.abs(arr), 95))
        # 不能低于硬编码最小值 15, 也不能高于 80. 低于 15 太敏感容易被噪声触发, 高于 80 基本上永远不会套利
        self.arb_thresh = float(np.clip(p95, 15.0, 80.0))
        print(f'[CALIB] arb_thresh={self.arb_thresh:.1f} (basis p95={p95:.1f})')

    def get_gamma(self, symbol):
        return self.gamma.get(symbol, self._gamma_default)

    def get_k(self, symbol):
        return self.k.get(symbol, self._k_default)


class AlgoBot(BaseBot):
    SYMBOLS = ['TIDE_SPOT','TIDE_SWING','WX_SPOT','WX_SUM','LHR_COUNT','LHR_INDEX','LON_ETF','LON_FLY']
    ETF_LEGS = ['TIDE_SPOT', 'WX_SPOT', 'LHR_COUNT']
    POSITION_LIMIT = 100
    SETTLE_TIME = datetime(2026, 3, 1, 12, 0, 0)

    MM_MIN_SPREAD = 3
    MM_BASE_QTY = 10
    REQUOTE_THRESH = 2
    ARB_QTY = 20
    ARB_BASIS_WINDOW = 50
    ARB_BASIS_STOP = 60
    DIR_Z_THRESH = 2.0
    DIR_QTY = 15
    DIR_DECAY_N = 3
    DIR_MIN_VOL_RATIO = 1.2
    DIR_MAX_SPREAD_RATIO = 2.0
    STOP_LOSS = -5000
    UNWIND_MINS = 30
    LOOP_INTERVAL = 10
    CORR_PAIRS = [('TIDE_SPOT','LON_ETF'), ('WX_SPOT','LON_ETF'), ('LHR_COUNT','LON_ETF'), ('LON_ETF','LON_FLY')]
    CORR_NET_LIMIT = 150
    FLY_KINK_POINTS = [6200, 6600, 7000]
    FLY_KINK_BAND = 50

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fve = FairValueEngine()
        self.calib  = ParamCalibrator()
        self.fair_values = {}
        self.market_mids = {}
        # 记录每个品种上一次报出去的 (bid, ask). 做市时用来判断新报价和旧报价差距是否超过 REQUOTE_THRESH=2, 没超过就不撤单重报, 减少无意义的撤单操作.
        self.last_quotes = {}
        self.active_orders = {}
        # 记录上次套利执行的时间戳. _realtime_arb_check做判断, 2 秒内不重复触发套利, 防止同一个basis信号被多次SSE推送反复执行
        self._arb_cooldown = 0.0
        # 记录每个品种上次方向性交易的时间戳. _realtime_dir_check做判断, 5秒内同一品种不重复触发, 防止SSE高频推送导致连续下单
        self._dir_cooldown = {}
        self._mid_history = {s: deque(maxlen=20) for s in self.SYMBOLS}
        self._vol_history = {s: deque(maxlen=20) for s in self.SYMBOLS}
        # 记录每个品种连续触发方向性信号的次数. 正数表示连续做空, 负数表示连续做多.
        # _execute_directional里用这个做衰减: 连续同向超过3次, 下单量减半, 防止单边行情中过度押注
        self._dir_streak = {s: 0 for s in self.SYMBOLS}
        self._basis_history = deque(maxlen=self.ARB_BASIS_WINDOW)
        self._arb_position = 0
        # 记录进入套利仓位时的basis值. _arb_basis_stop_check里用来判断basis是否朝不利方向移动超过60, 是则止损平仓. None表示当前没有套利仓位
        self._arb_entry_basis = None
        self._fly_delta = 0.0
        self._fly_gamma = 0.0
        # 记录上次对冲LON_FLY的时间戳. module_d_fly_hedge里用来控制对冲频率: 正常情况30秒对冲一次, ETF价格在折点附近时缩短到5秒
        self._last_hedge_ts = 0.0 
        # {symbol: spread} 用于校准器记录. on_orderbook里更新, on_trades 里读取, 传给 calib.record_fill
        self._last_spread = {}   

    # 每次市场上有成交时触发, 做两件事:
    # 把成交量记录进_vol_history, 用于判断市场活跃度
    # 把这次成交时的价差和量告诉校准器(calib.record_fill), 用于校准 k
    def on_trades(self, trade):
        side = 'BOUGHT' if trade.buyer == self.username else 'SOLD'
        self._vol_history[trade.product].append(trade.volume)
        spread = self._last_spread.get(trade.product, 0)
        self.calib.record_fill(trade.product, spread, trade.volume)
        print(f'FILL {side} {trade.volume}x {trade.product} @ {trade.price}')

    # 每次orderbook有变动时触发. 做四件事:
    # 更新market_mids和_mid_history
    # 更新_last_spread
    # 如果变动的是ETF或三条腿, 立即检查套利机会(_realtime_arb_check)
    # 如果变动的品种有FV, 立即检查方向性信号(_realtime_dir_check)
    def on_orderbook(self, ob):
        mid = self._calc_mid(ob)
        if mid is not None:
            self.market_mids[ob.product] = mid
            self._mid_history[ob.product].append(mid)
        spread = self._calc_spread(ob)
        if spread is not None:
            self._last_spread[ob.product] = spread
        if ob.product in self.ETF_LEGS + ['LON_ETF']:
            self._realtime_arb_check()
        if ob.product in self.fair_values:
            self._realtime_dir_check(ob.product)

    # 工具方法
    # 取orderbook的买一卖一中间价(best_bid + best_ask) / 2
    # 注意过滤掉自己的挂单(o.volume - o.own_volume > 0), 避免用自己的报价算中间价
    def _calc_mid(self, ob):
        bids = [o.price for o in ob.buy_orders if o.volume - o.own_volume > 0]
        asks = [o.price for o in ob.sell_orders if o.volume - o.own_volume > 0]
        return (max(bids) + min(asks)) / 2 if bids and asks else None

    # 取买卖价差best_ask - best_bid, 同样过滤自己的挂单
    def _calc_spread(self, ob):
        bids = [o.price for o in ob.buy_orders  if o.volume - o.own_volume > 0]
        asks = [o.price for o in ob.sell_orders if o.volume - o.own_volume > 0]
        return (min(asks) - max(bids)) if bids and asks else None

    # 检查orderbook买卖两侧是否各有足够的量(>= qty). 套利执行前用这个确认流动性, 避免下单后成交不了
    def _ob_depth(self, ob, qty):
        bid_vol = sum(o.volume - o.own_volume for o in ob.buy_orders)
        ask_vol = sum(o.volume - o.own_volume for o in ob.sell_orders)
        return bid_vol >= qty, ask_vol >= qty

    # 发送IOC单(Immediate-Or-Cancel). 下单后如果还是 ACTIVE 状态(没有完全成交), 立即撤销. 用于套利和方向性交易, 确保不会留下挂单
    def _send_ioc(self, order):
        resp = self.send_order(order)
        if resp and resp.status == 'ACTIVE':
            self.cancel_order(resp.id)
        return resp
        
    # 撤销某个品种的所有挂单, 并清空active_orders记录. 做市重新报价前调用
    def _cancel_symbol(self, symbol):
        for oid in self.active_orders.get(symbol, []):
            self.cancel_order(oid)
        self.active_orders[symbol] = []

    # 返回距结算还有多少分钟, 驱动T的计算和平仓逻辑
    def _mins_to_settle(self):
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        now = datetime.now(pytz.timezone('Europe/London'))
        return (settle - now).total_seconds() / 60

    def _get_positions(self):
        return self.get_positions()

    # 用最近20条mid价格的标准差估算短期波动率. 数据不足3条时用FV的std兜底. 这个值传给校准器的record_vol
    def _short_vol(self, symbol):
        hist = list(self._mid_history[symbol])
        if len(hist) < 3:
            return self.fair_values.get(symbol, (0, self.MM_MIN_SPREAD * 2))[1]
        return max(float(np.std(hist)), self.MM_MIN_SPREAD)

    # 返回最近成交量的均值, 用于判断市场是否活跃
    def _recent_trade_vol(self, symbol):
        hist = list(self._vol_history[symbol])
        return float(np.mean(hist)) if hist else 0.0
    
    # 模块 A: A-S做市(使用校准后的 gamma/k) ---------------------------------------------------------------------------------------
    # 从校准器拿gamma和k
    # 算reservation price: r = fv - q × gamma × sigma^2 × T, 持仓越多报价越偏离中心, 主动减仓
    # 算最优半价差: half_spread = (gamma×sigma^2×T + (2/gamma)×ln(1+gamma/k)) / 2
    # 如果新报价和上次差距小于 REQUOTE_THRESH=2, 跳过, 避免频繁撤单重报
    # 撤掉旧单, 按仓位上限约束挂新的买卖单
    # 注意: 单一品种的做市仓位风险是靠 A-S 模型本身来管理的, 不是靠对冲. 具体机制是 reservation price 的偏移
    def module_a_market_make(self, positions):
        mins_left = self._mins_to_settle()
        T = max(mins_left / (24 * 60), 1e-4)
        for symbol in self.SYMBOLS:
            # 向校准器记录当前波动率并使用校准后的参数
            if symbol not in self.fair_values:
                continue
            fv, model_std = self.fair_values[symbol]
            sigma = self._short_vol(symbol)
            self.calib.record_vol(symbol, sigma)
            pos = positions.get(symbol, 0)
            q = pos / self.POSITION_LIMIT
            gamma = self.calib.get_gamma(symbol)
            k = self.calib.get_k(symbol)

            # reservation price, 即内心真正愿意成交的极限价格
            # 如空头重仓, 此时我们想赶紧买货补库存, 别再卖更多货, 所以reservation price > fair price
            r = fv - q * gamma * sigma**2 * T
            
            # 最优spread
            as_spread = (gamma * sigma**2 * T+ (2.0 / gamma) * np.log(1 + gamma / k))
            half_spread = max(self.MM_MIN_SPREAD, as_spread / 2)
            bid_price = round(r - half_spread)
            ask_price = round(r + half_spread)
            if bid_price <= 0 or ask_price <= bid_price:
                continue

            # 避免频繁撤单重报
            last = self.last_quotes.get(symbol)
            if (last and abs(last[0] - bid_price) < self.REQUOTE_THRESH and abs(last[1] - ask_price) < self.REQUOTE_THRESH):
                continue

            # 撤掉旧单, 按仓位上限约束挂新的买卖单
            self._cancel_symbol(symbol)
            avail_buy = max(0, self.POSITION_LIMIT - pos)
            avail_sell = max(0, self.POSITION_LIMIT + pos)
            new_ids = []
            if avail_buy >= 1:
                qty = min(self.MM_BASE_QTY, avail_buy)
                r2 = self.send_order(OrderRequest(symbol, bid_price, Side.BUY, qty))
                if r2: 
                    new_ids.append(r2.id)
            if avail_sell >= 1:
                qty = min(self.MM_BASE_QTY, avail_sell)
                r2 = self.send_order(OrderRequest(symbol, ask_price, Side.SELL, qty))
                if r2: 
                    new_ids.append(r2.id)
            self.active_orders[symbol] = new_ids
            self.last_quotes[symbol] = (bid_price, ask_price)

    # 模块 B: ETF套利(使用校准后的 arb_thresh) ---------------------------------------------------------------------------------------
    # SSE 触发的实时套利检查. 计算 basis, 超过动态阈值就执行套利. 有 2 秒冷却防止重复触发
    def _realtime_arb_check(self):
        if time.time() - self._arb_cooldown < 2.0:
            return
        etf_mid = self.market_mids.get('LON_ETF')
        leg_mids = [self.market_mids.get(s) for s in self.ETF_LEGS]
        if etf_mid is None or any(m is None for m in leg_mids):
            return
        basis = etf_mid - sum(leg_mids)
        self._basis_history.append(basis)
        self.calib.record_basis(basis)
        thresh = self.calib.arb_thresh
        if abs(basis) > thresh:
            positions = self._get_positions()
            self._execute_arb(basis, positions)
            self._arb_cooldown = time.time()

    # 套利止损. 如果已有套利仓位, 且basis朝不利方向移动超过 ARB_BASIS_STOP=60, 立即平仓. 防止 basis 持续扩大导致亏损。
    def _arb_basis_stop_check(self, positions):
        if self._arb_position == 0 or self._arb_entry_basis is None:
            return
        etf_mid = self.market_mids.get('LON_ETF')
        leg_mids = [self.market_mids.get(s) for s in self.ETF_LEGS]
        if etf_mid is None or any(m is None for m in leg_mids):
            return
        current_basis = etf_mid - sum(leg_mids)
        # 进场时 basis 是正的, 赌 basis 会收窄回零, 结果 basis 不但没收窄, 反而继续扩大超过 60.
        # 进场时 basis 是负的, 赌 basis 会收窄回零, 结果 basis 不但没收窄, 反而继续扩大超过 60.
        adverse = (
            (self._arb_position > 0 and current_basis > self._arb_entry_basis + self.ARB_BASIS_STOP) or
            (self._arb_position < 0 and current_basis < self._arb_entry_basis - self.ARB_BASIS_STOP)
        )
        if adverse:
            print(f'ARB BASIS STOP: entry={self._arb_entry_basis:.1f} current={current_basis:.1f}')
            self._close_arb_position(positions)
            self._arb_position = 0
            self._arb_entry_basis = None

    # 强制平掉ETF和三条腿的所有套利仓位, 用IOC单以市价成交。
    def _close_arb_position(self, positions):
        etf_pos = positions.get('LON_ETF', 0)
        if etf_pos != 0:
            ob = self.get_orderbook('LON_ETF')
            if etf_pos > 0 and ob.buy_orders:
                self._send_ioc(OrderRequest('LON_ETF', max(o.price for o in ob.buy_orders), Side.SELL, abs(etf_pos)))
            elif etf_pos < 0 and ob.sell_orders:
                self._send_ioc(OrderRequest('LON_ETF', min(o.price for o in ob.sell_orders), Side.BUY, abs(etf_pos)))
        for s in self.ETF_LEGS:
            leg_pos = positions.get(s, 0)
            if leg_pos == 0: continue
            ob = self.get_orderbook(s)
            if leg_pos > 0 and ob.buy_orders:
                self._send_ioc(OrderRequest(s, max(o.price for o in ob.buy_orders), Side.SELL, abs(leg_pos)))
            elif leg_pos < 0 and ob.sell_orders:
                self._send_ioc(OrderRequest(s, min(o.price for o in ob.sell_orders), Side.BUY, abs(leg_pos)))

    # 实际执行套利：
    # basis > 0(ETF偏贵): 卖 ETF + 买三条腿
    # basis < 0(ETF偏便宜): 买 ETF + 卖三条腿
    # 执行前检查仓位上限和 orderbook 深度，四条腿都有足够流动性才下单。
    def _execute_arb(self, basis, positions):
        if basis > 0:
            qty = min(self.ARB_QTY, self.POSITION_LIMIT + positions.get('LON_ETF', 0),
                      min(self.POSITION_LIMIT - positions.get(s, 0) for s in self.ETF_LEGS))
            if qty < 1: 
                return
            ob_etf = self.get_orderbook('LON_ETF')
            bid_ok, _ = self._ob_depth(ob_etf, qty)
            if not bid_ok: 
                return
            leg_obs = {s: self.get_orderbook(s) for s in self.ETF_LEGS}
            for s, ob in leg_obs.items():
                _, ask_ok = self._ob_depth(ob, qty)
                if not ask_ok: 
                    return
            print(f' ARB SELL ETF basis={basis:.1f} qty={qty}')
            if ob_etf.buy_orders:
                self._send_ioc(OrderRequest('LON_ETF', max(o.price for o in ob_etf.buy_orders), Side.SELL, qty))
            for s, ob in leg_obs.items():
                if ob.sell_orders:
                    self._send_ioc(OrderRequest(s, min(o.price for o in ob.sell_orders), Side.BUY, qty))
            self._arb_position = 1
            self._arb_entry_basis = basis
        else:
            qty = min(self.ARB_QTY, self.POSITION_LIMIT - positions.get('LON_ETF', 0),
                      min(self.POSITION_LIMIT + positions.get(s, 0) for s in self.ETF_LEGS))
            if qty < 1: 
                return
            ob_etf = self.get_orderbook('LON_ETF')
            _, ask_ok = self._ob_depth(ob_etf, qty)
            if not ask_ok: 
                return
            leg_obs = {s: self.get_orderbook(s) for s in self.ETF_LEGS}
            for s, ob in leg_obs.items():
                bid_ok, _ = self._ob_depth(ob, qty)
                if not bid_ok: 
                    return
            print(f' ARB BUY ETF basis={basis:.1f} qty={qty}')
            if ob_etf.sell_orders:
                self._send_ioc(OrderRequest('LON_ETF', min(o.price for o in ob_etf.sell_orders), Side.BUY, qty))
            for s, ob in leg_obs.items():
                if ob.buy_orders:
                    self._send_ioc(OrderRequest(s, max(o.price for o in ob.buy_orders), Side.SELL, qty))
            self._arb_position = -1
            self._arb_entry_basis = basis

    # 主循环调用的入口. 先检查止损, 再检查新的套利机会
    # SSE 推送 → on_orderbook → _realtime_arb_check → _execute_arb
    # 主循环每10秒 → module_b_arbitrage → _execute_arb
    def module_b_arbitrage(self, positions):
        self._arb_basis_stop_check(positions)
        etf_mid = self.market_mids.get('LON_ETF')
        leg_mids = [self.market_mids.get(s) for s in self.ETF_LEGS]
        if etf_mid is None or any(m is None for m in leg_mids):
            return
        basis = etf_mid - sum(leg_mids)
        self._basis_history.append(basis)
        self.calib.record_basis(basis)
        if abs(basis) > self.calib.arb_thresh:
            self._execute_arb(basis, positions)

    # 模块 C: 方向性 --------------------------------------------------------------------------------------------------------------------
    # SSE触发的实时方向性检查. 计算 z-score, 超过阈值就执行. 有 5 秒冷却.
    def _realtime_dir_check(self, symbol):
        if time.time() - self._dir_cooldown.get(symbol, 0) < 5.0:
            return
        if symbol not in self.fair_values:
            return
        fv, std = self.fair_values[symbol]
        mid = self.market_mids.get(symbol)
        if mid is None or std <= 0:
            return
        z = (mid - fv) / std
        if abs(z) > self.DIR_Z_THRESH:
            positions = self._get_positions()
            self._execute_directional(symbol, z, fv, mid, std, positions)
            self._dir_cooldown[symbol] = time.time()
        
    # 信号质量过滤器, 两个条件: 
    # 当前价差不能超过正常价差的2倍(价差太宽说明流动性差，不适合吃单)
    # 最近一笔成交量要大于历史均值的1.2倍(有成交量放大才说明信号可信)
    def _dir_signal_valid(self, symbol):
        try:
            ob = self.get_orderbook(symbol)
            cur_spread = self._calc_spread(ob)
            fv, std = self.fair_values.get(symbol, (0, self.MM_MIN_SPREAD * 2))
            normal_spread = max(std * 0.5, self.MM_MIN_SPREAD)
            if cur_spread is not None and cur_spread > normal_spread * self.DIR_MAX_SPREAD_RATIO:
                return False
        except Exception:
            pass
        avg_vol = self._recent_trade_vol(symbol)
        if avg_vol < 1.0:
            return True
        recent = list(self._vol_history[symbol])
        if not recent:
            return True
        return recent[-1] >= avg_vol * self.DIR_MIN_VOL_RATIO

    # 执行方向性交易. 有一个衰减机制: 连续同向交易超过DIR_DECAY_N=3次后, 下单量减半, 再连续3次再减半. 防止在单边行情中过度押注.
    def _execute_directional(self, symbol, z, fv, mid, std, positions):
        if not self._dir_signal_valid(symbol):
            return
        pos = positions.get(symbol, 0)
        streak = self._dir_streak.get(symbol, 0)
        base_qty = self.DIR_QTY
        # 衰减机制
        if z > 0 and streak >= self.DIR_DECAY_N:
            base_qty = max(1, base_qty // (2 ** (streak // self.DIR_DECAY_N)))
        elif z < 0 and streak <= -self.DIR_DECAY_N:
            base_qty = max(1, base_qty // (2 ** (abs(streak) // self.DIR_DECAY_N)))
        # 下单
        if z > self.DIR_Z_THRESH:
            qty = min(base_qty, self.POSITION_LIMIT + pos)
            if qty < 1: 
                self._dir_streak[symbol] = 0
                return
            ob = self.get_orderbook(symbol)
            if ob.buy_orders:
                print(f' DIR SELL {symbol} z={z:.2f} qty={qty} streak={streak}')
                self._send_ioc(OrderRequest(symbol, max(o.price for o in ob.buy_orders), Side.SELL, qty))
            self._dir_streak[symbol] = max(0, streak) + 1
        elif z < -self.DIR_Z_THRESH:
            qty = min(base_qty, self.POSITION_LIMIT - pos)
            if qty < 1: 
                self._dir_streak[symbol] = 0
                return
            ob = self.get_orderbook(symbol)
            if ob.sell_orders:
                print(f'DIR BUY  {symbol} z={z:.2f} qty={qty} streak={streak}')
                self._send_ioc(OrderRequest(symbol, min(o.price for o in ob.sell_orders), Side.BUY, qty))
            self._dir_streak[symbol] = min(0, streak) - 1
        else:
            self._dir_streak[symbol] = 0

    # 主循环调用的入口, 遍历所有品种检查 z-score.
    # SSE 推送 → on_orderbook → _realtime_dir_check → _dir_signal_valid → _execute_directional
    # 主循环每10秒 → module_c_directional → _dir_signal_valid → _execute_directional
    def module_c_directional(self, positions):
        for symbol in self.SYMBOLS:
            if symbol not in self.fair_values: 
                continue
            fv, std = self.fair_values[symbol]
            mid = self.market_mids.get(symbol)
            if mid is None or std <= 0: 
                continue
            z = (mid - fv) / std
            if abs(z) > self.DIR_Z_THRESH:
                self._execute_directional(symbol, z, fv, mid, std, positions)

    # 模块 D: LON_FLY delta+gamma 对冲 ---------------------------------------------------------------------------------------
    # 检查ETF价格是否在折点附近(±50). 折点附近delta变化快, 对冲频率加倍(冷却从30秒缩到5秒), 阈值从5缩到2
    # 用fve.lon_fly_greeks()算当前delta, 失败时用分段线性的近似值兜底
    # 计算目标对冲量: target_hedge = -fly_pos × delta
    # 和当前ETF仓位比较, 差距超过阈值才下单调整
    def module_d_fly_hedge(self, positions):
        fly_pos = positions.get('LON_FLY', 0)
        if fly_pos == 0: 
            return
        etf_mid = self.market_mids.get('LON_ETF')
        if etf_mid is None: 
            return
        near_kink = any(abs(etf_mid - k) < self.FLY_KINK_BAND for k in self.FLY_KINK_POINTS)
        hedge_threshold = 2 if near_kink else 5
        hedge_cooldown  = 5.0 if near_kink else 30.0
        if time.time() - self._last_hedge_ts < hedge_cooldown:
            return
        try:
            delta, gamma = self.fve.lon_fly_greeks()
            self._fly_delta = delta; self._fly_gamma = gamma
        except Exception:
            if   etf_mid < 6200: 
                self._fly_delta = -2.0
            elif etf_mid < 6600: 
                self._fly_delta = -1.0
            elif etf_mid < 7000: 
                self._fly_delta =  1.0
            else:                
                self._fly_delta = -2.0
        target_hedge = -fly_pos * self._fly_delta
        current_etf  = positions.get('LON_ETF', 0)
        delta_needed = target_hedge - current_etf
        if abs(delta_needed) < hedge_threshold: 
            return
        qty = min(abs(int(delta_needed)), self.POSITION_LIMIT - abs(current_etf))
        if qty < 1: 
            return
        ob = self.get_orderbook('LON_ETF')
        if delta_needed > 0 and ob.sell_orders:
            print(f' HEDGE BUY LON_ETF {qty} (near_kink={near_kink})')
            self._send_ioc(OrderRequest('LON_ETF', min(o.price for o in ob.sell_orders), Side.BUY, qty))
        elif delta_needed < 0 and ob.buy_orders:
            print(f' HEDGE SELL LON_ETF {qty} (near_kink={near_kink})')
            self._send_ioc(OrderRequest('LON_ETF', max(o.price for o in ob.buy_orders), Side.SELL, qty))
        self._last_hedge_ts = time.time()

    # 模块 E: 组合风险 ---------------------------------------------------------------------------------------
    # 打印当前组合名义价值和FLY的 delta/gamma, 方便监控
    # 检查相关品种对的总仓位. CORR_PAIRS里定义了4对相关品种, 如果两者仓位之和超过CORR_NET_LIMIT=150, 强制减仓较大的那个
    # 这是防止相关品种同向押注过多, 一旦方向判断错误损失会被放大.
    def module_e_portfolio_risk(self, positions):
        total_notional = sum(abs(pos) * (self.fair_values.get(sym, (0,0))[0] or 0) for sym, pos in positions.items())
        print(f'PORTFOLIO notional~{total_notional:.0f} fly_delta={self._fly_delta:.2f} fly_gamma={self._fly_gamma:.4f}')
        for sym_a, sym_b in self.CORR_PAIRS:
            pos_a = positions.get(sym_a, 0)
            pos_b = positions.get(sym_b, 0)
            net = abs(pos_a) + abs(pos_b)
            if net > self.CORR_NET_LIMIT:
                target = sym_a if abs(pos_a) >= abs(pos_b) else sym_b
                tpos = positions.get(target, 0)
                rqty = min(abs(tpos), net - self.CORR_NET_LIMIT)
                if rqty < 1: 
                    continue
                ob = self.get_orderbook(target)
                if tpos > 0 and ob.buy_orders:
                    self._send_ioc(OrderRequest(target, max(o.price for o in ob.buy_orders), Side.SELL, rqty))
                elif tpos < 0 and ob.sell_orders:
                    self._send_ioc(OrderRequest(target, min(o.price for o in ob.sell_orders), Side.BUY, rqty))

    # 结算前平仓 ---------------------------------------------------------------------------------------
    # 距结算30分钟内触发. 撤所有挂单, 然后对每个有仓位的品种用IOC市价单强制平仓. 确保结算时没有净仓位.
    def unwind_all(self, positions):
        print('UNWIND: closing all positions before settlement')
        self.cancel_all_orders()
        for symbol, pos in positions.items():
            if pos == 0: continue
            ob = self.get_orderbook(symbol)
            if pos > 0 and ob.buy_orders:
                self._send_ioc(OrderRequest(symbol, max(o.price for o in ob.buy_orders), Side.SELL, abs(pos)))
            elif pos < 0 and ob.sell_orders:
                self._send_ioc(OrderRequest(symbol, min(o.price for o in ob.sell_orders), Side.BUY, abs(pos)))

    # 主循环 ---------------------------------------------------------------------------------------
    # 启动 → init_flights → start SSE
    # 每轮:
    # 检查是否进入平仓期(< 30分钟)→ 是则unwind_all
    # 主动轮询所有品种orderbook, 更新market_mids
    # fve.get_all()计算贝叶斯后验fair_values
    # 检查止损(unrealizedPnl < -5000 则停止)
    # calib.tick()触发参数校准
    # 依次执行E → B → C → D → A五个模块
    # sleep补足10秒间隔
    def run_bot(self):
        print('Initialising...')
        self.fve.init_flights()
        self.start()
        print('Bot running. Ctrl+C to stop.')
        try:
            while True:
                # 检查是否进入平仓期
                t0 = time.time()
                mins_left = self._mins_to_settle()

                if mins_left <= self.UNWIND_MINS:
                    positions = self._get_positions()
                    self.unwind_all(positions)
                    print(f'{mins_left:.1f} mins to settle, unwinding. Sleeping 60s.')
                    time.sleep(60)
                    continue

                # 更新orderbook
                for sym in self.SYMBOLS:
                    try:
                        ob = self.get_orderbook(sym)
                        mid = self._calc_mid(ob)
                        if mid:
                            self.market_mids[sym] = mid
                            self._mid_history[sym].append(mid)
                    except Exception:
                        pass
                        
                # 计算fair_values
                self.fair_values = self.fve.get_all(market_mids=self.market_mids)
                fv_str = {s: f'{v[0]}+-{v[1]:.0f}' for s, v in self.fair_values.items()}
                print(f'\n[{datetime.now().strftime("%H:%M:%S")}] {mins_left:.0f}min left')
                print(f'FV: {fv_str}')

                # 检查止损
                positions = self._get_positions()
                pnl = self.get_pnl()
                print(f'Pos: {positions}  PnL: {pnl}')
                if pnl.get('unrealizedPnl', 0) < self.STOP_LOSS:
                    print(f'  STOP LOSS triggered at {pnl}')
                    self.cancel_all_orders(); self.stop(); 
                    break

                # 参数gamma, k, arb_thresh自动校准(每CALIB_INTERVAL 轮触发一次)
                self.calib.tick(self.SYMBOLS, self.fair_values, mins_left)

                # E module_e_portfolio_risk: 检查组合风险, 相关品种仓位超限就强制减仓
                # B module_b_arbitrage: 检查ETF basis, 超过阈值就套利, 同时检查套利止损
                # C module_c_directional: 检查z-score, 市场价偏离FV超过2个标准差就赌方向
                # D module_d_fly_hedge: 检查LON_FLY仓位的delta敞口, 用LON_ETF对冲
                # A module_a_market_make: 用A-S模型计算最优报价，挂买卖单赚价差
                # 风险控制优先于一切, E放最前面, 确保在做任何交易之前先把组合风险控制在合理范围内. 如果仓位已经超限, 后面的模块就不应该再加仓.
                # 主动交易在做市之前，B/C/D在A之前, 是因为套利和方向性交易会改变仓位, 仓位变了之后A-S模型的reservation price也会变.
                # 如果先挂做市单再做套利, 做市单的价格就是基于错误仓位算出来的.
                # 对冲在做市之前, D在A之前. 同理, LON_FLY的delta对冲会改变LON_ETF仓位，对冲完再报LON_ETF的做市价才准确。
                # 真实的量化交易系统基本都遵循这个框架: 风险检查 → 主动策略 → 被动做市。
                self.module_e_portfolio_risk(positions)
                positions = self._get_positions()  # 重新拉仓位, 反映module_e的减仓结果
                self.module_b_arbitrage(positions)
                self.module_c_directional(positions)
                self.module_d_fly_hedge(positions)
                self.module_a_market_make(positions)

                # sleep补足10秒间隔
                elapsed = time.time() - t0
                time.sleep(max(1, self.LOOP_INTERVAL - elapsed))

        except KeyboardInterrupt:
            print('\nStopping...')
            self.cancel_all_orders(); self.stop()
            print('Final positions:', self.get_positions())
            print('Final PnL:', self.get_pnl())
        except Exception as e:
            import traceback
            traceback.print_exc()
            self.cancel_all_orders(); self.stop()


In [ ]:
bot = AlgoBot(EXCHANGE_URL, USERNAME, PASSWORD)
bot.run_bot()